<!-- nav -->
[← 8.14 · SystemVerilog Patterns](08_14_SystemVerilog_Patterns.ipynb) · [Home](../../README.md) · [8.16 · SystemVerilog Constraints →](08_16_SystemVerilog_Constraints.ipynb)

# 8.15 · SystemVerilog Assertions

SystemVerilog Assertions describe behavior over clock cycles (IEEE 1800 clause 16): a **sequence** says what happens
from one cycle to the next, `a ##1 b`, `a` then `b` a cycle later; a **property** says what must hold of sequences,
`a |=> b`, whenever `a`, then `b` in the next cycle. Tutorial 8 showed one of each (08.4). This page builds them up in
their dialect, `SystemVerilog Assertions`: delays and cycle ranges, repetitions and the sequence operators, the
property operators, a property's clock and reset, each in the builders' fluent calls beside the text SystemVerilog
writes, the printer placing parentheses by the standard's precedence. Then what is done with them: held by a call's
argument, validated, with one refused, and never simplified, though the expressions among them are; last, the editions
that have each kind. The statement that asserts a property, `assert property (...)`, is mbse-programs'.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4), which are a sequence's
simplest operands, and `refused` gives what a simplifier says of an expression, its value where its residual is a literal, or why it has none, as tutorial 8's does.

In [1]:
from mbse.Expressions.SystemVerilog import Assertions as SVAssertions, Expressions as SystemVerilog
from mbse.Expressions.Framework import Domains, Standards, Symbolics, Terms
from toolkit import dialects

store = dialects()
S = store.Expressions
Assertions = SVAssertions.Expressions.DIALECT  # the dialect, as the framework reads it
name = lambda spelling: lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling(spelling))))
literal = lambda value: lambda d: d.literal(lambda l: l.value(lambda v: v.as_scalar(value)))
a, b, c, en, n = name("a"), name("b"), name("c"), name("en"), name("n")



def refused(simplify, expression):
    """What `simplify` says of `expression`: its value, a simplification's where its residual is a literal, or why it has
    none (`NotEvaluated`, a `ValueError`)."""
    try:
        result = simplify(expression)
        return str(result.evaluate() if hasattr(result, "evaluate") else result)
    except ValueError as error:
        return str(error)

<a id="T08.15.1"></a>
## 08.15.1 · A sequence of delays

```text
a ##1 b        a ##1 b ##[1:$] c        ##2 a
```

`a ##1 b` is a `delaysequence`: its `first` operand, `a`, then its `steps`, each a `DelayStep` helper holding a delay and
the sequence that follows it. A delay is an expression, `1`, or a cycle range, `[1:$]`, a `cyclerange` from one cycle to
any number (`$`, `Expressions`' `dollarexpression`); a sequence that starts with a delay has no `first`. Its simplest
operands are expressions: a name is a sequence that holds in the cycles where it is true. Its form gives its terms in
order, `first`, then each step's delay and sequence, two by two:

In [2]:
with store.transaction():
    delays = [S.SystemVerilog.Assertions.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.delaysequence(lambda s: s.first(a).steps([lambda t: t.delay(literal(1)).sequence(b)])),
        lambda u: u.delaysequence(lambda s: s.first(a).steps([
            lambda t: t.delay(literal(1)).sequence(b),
            lambda t: t.delay(lambda d: d.cyclerange(lambda r: r.low(literal(1)).high(lambda d: d.dollarexpression(lambda r: r)))).sequence(c)])),
        lambda u: u.delaysequence(lambda s: s.steps([lambda t: t.delay(literal(2)).sequence(a)])),
    ]]
for sequence in delays:
    form = Terms.form(Terms.term(sequence))
    print(f"{SystemVerilog.Text.ToText(sequence):19}", form.kind, "| its terms:",
          ", ".join(Terms.kind_of(t)[0] if t is not None else "-" for t in form.arguments))
assert [SystemVerilog.Text.ToText(s) for s in delays] == ["a ##1 b", "a ##1 b ##[1:$] c", "##2 a"]

a ##1 b             delaysequence | its terms: nameexpression, literal, nameexpression
a ##1 b ##[1:$] c   delaysequence | its terms: nameexpression, literal, nameexpression, cyclerange, nameexpression
##2 a               delaysequence | its terms: -, literal, nameexpression


The empty `first` of `##2 a` is a `-` among its terms: a subterm left unset, as a lambda's `value` was in tutorial 7.

<a id="T08.15.2"></a>
## 08.15.2 · Repetitions and the sequence operators

```text
a[*2]    b[->1:3]    c[=n]    a and b    en throughout b    first_match(a or b)    a ##1 (b or c)
```

A repetition, `repetitionsequence`, repeats a sequence consecutively (`[*2]`), to its last match (`[->1:3]`, a goto
repetition) or not consecutively (`[=n]`), its count an expression or a cycle range. Two sequences combine by
`binarysequence`, its operator `and`, `or`, `intersect`, `within` or `throughout`; `first_match` keeps a sequence's
first match. Which operator binds tighter is the standard's (its Table 16-3), so a tree whose `##1` takes `b or c` as
its sequence prints parentheses the tree does not hold, as tutorial 5 showed for expressions:

In [3]:
with store.transaction():
    operators = [S.SystemVerilog.Assertions.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.repetitionsequence(lambda r: r.sequence(a).operator("*").count(literal(2))),
        lambda u: u.repetitionsequence(lambda r: r.sequence(b).operator("->").count(lambda d: d.cyclerange(lambda r: r.low(literal(1)).high(literal(3))))),
        lambda u: u.repetitionsequence(lambda r: r.sequence(c).operator("=").count(n)),
        lambda u: u.binarysequence(lambda s: s.left(a).operator("and").right(b)),
        lambda u: u.binarysequence(lambda s: s.left(en).operator("throughout").right(b)),
        lambda u: u.firstmatchsequence(lambda f: f.sequence(lambda u: u.binarysequence(lambda s: s.left(a).operator("or").right(b)))),
        lambda u: u.delaysequence(lambda s: s.first(a).steps([lambda t: t.delay(literal(1)).sequence(lambda u: u.binarysequence(lambda s: s.left(b).operator("or").right(c)))])),
    ]]
for sequence in operators:
    print(f"{SystemVerilog.Text.ToText(sequence):20}", Terms.kind_of(Terms.term(sequence))[0])
assert [SystemVerilog.Text.ToText(s) for s in operators] == [
    "a[*2]", "b[->1:3]", "c[=n]", "a and b", "en throughout b", "first_match(a or b)", "a ##1 (b or c)"]

a[*2]                repetitionsequence
b[->1:3]             repetitionsequence
c[=n]                repetitionsequence
a and b              binarysequence
en throughout b      binarysequence
first_match(a or b)  firstmatchsequence
a ##1 (b or c)       delaysequence


<a id="T08.15.3"></a>
## 08.15.3 · Properties

```text
a |-> b    a |=> b    always a    s_eventually b    not (a and b)    nexttime [2] a    if (en) a else b
```

A property is built of sequences: an implication, `implicationproperty`, whose consequent must hold from the cycle its
antecedent matches (`|->`) or the next (`|=>`); a unary operator, `unaryproperty`, `not`, `always`, `s_eventually`,
`nexttime` and the rest, perhaps with a range of cycles; a binary one, `binaryproperty` (`and`, `or`, `implies`,
`until`, ...); and a conditional, `if (en) a else b`. Where a sequence may stand, so may a property, and an expression,
so a property's operands are this dialect's terms:

In [4]:
with store.transaction():
    properties = [S.SystemVerilog.Assertions.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.implicationproperty(lambda p: p.antecedent(a).operator("|->").consequent(b)),
        lambda u: u.implicationproperty(lambda p: p.antecedent(a).operator("|=>").consequent(b)),
        lambda u: u.unaryproperty(lambda p: p.operator("always").operand(a)),
        lambda u: u.unaryproperty(lambda p: p.operator("s_eventually").operand(b)),
        lambda u: u.unaryproperty(lambda p: p.operator("not").operand(lambda u: u.binaryproperty(lambda p: p.left(a).operator("and").right(b)))),
        lambda u: u.unaryproperty(lambda p: p.operator("nexttime").range(literal(2)).operand(a)),
        lambda u: u.conditionalproperty(lambda p: p.condition(en).consequence(a).alternative(b)),
    ]]
for property_ in properties:
    print(f"{SystemVerilog.Text.ToText(property_):17}", Terms.kind_of(Terms.term(property_))[0])
assert [SystemVerilog.Text.ToText(p) for p in properties] == [
    "a |-> b", "a |=> b", "always a", "s_eventually b", "not (a and b)", "nexttime [2] a", "if (en) a else b"]

a |-> b           implicationproperty
a |=> b           implicationproperty
always a          unaryproperty
s_eventually b    unaryproperty
not (a and b)     unaryproperty
nexttime [2] a    unaryproperty
if (en) a else b  conditionalproperty


<a id="T08.15.4"></a>
## 08.15.4 · A property's clock and reset

```text
@(posedge clk) disable iff (rst) req |=> ack ##[1:3] done
```

A property is checked on a clock and abandoned while a reset holds: a `propertyspec` holds its `clock`, an
`EventControl` helper of `EventExpression`s (each an edge, an expression and perhaps an `iff` condition), its `disable`
condition, an expression, and its `property`. The clock is a helper, so the form gives it as a part; its signal is a
term all the same, and `Symbolics.free` finds every name the property reads, the clock's among them:

In [5]:
with store.transaction():
    handshake = S.SystemVerilog.Assertions.Expression(lambda e: e.root(lambda u: u.propertyspec(lambda s: s
        .clock(lambda k: k.events([lambda v: v.edge("posedge").expression(name("clk"))]))
        .disable(name("rst"))
        .property(lambda u: u.implicationproperty(lambda p: p.antecedent(name("req")).operator("|=>").consequent(lambda d: d.delaysequence(lambda q: q.first(name("ack")).steps([
            lambda t: t.delay(lambda d: d.cyclerange(lambda r: r.low(literal(1)).high(literal(3)))).sequence(name("done"))]))))))))
form = Terms.form(Terms.term(handshake))
print(SystemVerilog.Text.ToText(handshake))
print(form.kind, "| clock:", form.attributes["clock"].kind, "| its terms:",
      ", ".join(Terms.kind_of(t)[0] if t is not None else "-" for t in form.arguments))
print("free:", ", ".join(sorted(Symbolics.free(handshake, Assertions))))
assert SystemVerilog.Text.ToText(handshake) == "@(posedge clk) disable iff (rst) req |=> ack ##[1:3] done"

@(posedge clk) disable iff (rst) req |=> ack ##[1:3] done
propertyspec | clock: eventcontrol | its terms: nameexpression, implicationproperty, nameexpression, -
free: ack, clk, done, req, rst


<a id="T08.15.5"></a>
## 08.15.5 · Held by a call, validated, and one refused

```text
f(a ##1 b)
```

A named sequence or property is given its arguments as a call is (16.8): a call's arguments, and a named connection's
value, hold this dialect's `Term`, the properties of `Expressions`' kinds read from the store as tutorial 8 did (08.3). So
`f(a ##1 b)` is `Expressions`' call, its argument an Assertions term, and its host validates it, reading it through the
dialects it reads. `validate` checks each kind's operator among its choices: `->` is no implication's, and is refused:

In [6]:
union = store.registered(Assertions.union)
host = store.registered(SystemVerilog.Expressions.DIALECT.union).branches
print(", ".join(f"{b.name}.{p}" for b in host for p, prop in b.schema.properties.items()
                if union in (prop.schema, getattr(prop.schema, "element", None))))
with store.transaction():
    called = S.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda u: u.callexpression(lambda f: f
        .callee(name("f")).arguments([lambda d: d.delaysequence(lambda s: s.first(a).steps([lambda t: t.delay(literal(1)).sequence(b)]))]))))
argument = Terms.form(Terms.term(called)).arguments[1]
dialect = SystemVerilog.Expressions.DIALECT
print(SystemVerilog.Text.ToText(called), "| its argument:", Terms.kind_of(argument)[0], "of", dialect.owner(argument).name(),
      "| problems:", "; ".join(dialect.validate(called, {"f", "a"})) or "none")
with store.transaction():
    arrow = S.SystemVerilog.Assertions.Expression(lambda e: e.root(lambda u: u.implicationproperty(lambda p: p.antecedent(a).operator("->").consequent(b))))
print(SystemVerilog.Text.ToText(arrow), "|", "; ".join(Assertions.validate(arrow, {"a", "b"})))
assert Assertions.validate(arrow, {"a", "b"}) == ["ImplicationProperty.operator cannot be '->'"]

callexpression.arguments, namedconnection.value
f(a ##1 b) | its argument: delaysequence of SystemVerilog Assertions | problems: argument 0: sequence 0: nameexpression 'b' is not bound
a -> b | ImplicationProperty.operator cannot be '->'


Given `f` and `a` alone, the host reports the `b` the sequence reads, by where it stands: the call's argument 0, its
first step's sequence, `sequence 0`.

<a id="T08.15.6"></a>
## 08.15.6 · Never simplified, but its expressions are

SystemVerilog's rules simulate no clock cycles: they fold none of the dialect's kinds: each stays in the residual, named (stage 6, "Q11"),
held alone or as a call's argument, and an Assertions term has no domain to infer, `any`. An expression is no
assertion kind, though, and held by the same holder it still folds, `a + n` to a value of `integer` (tutorial 3,
03.5):

In [7]:
simplify = SystemVerilog.Simplifiers.OfAny
values = {"a": 1, "b": 0, "n": 2}
for expression in [delays[0], operators[3], properties[1], handshake]:
    print(f"{SystemVerilog.Text.ToText(expression):30}", refused(lambda e: simplify(e, values), expression))
with store.transaction():
    total = S.SystemVerilog.Assertions.Expression(lambda e: e.root(lambda u: u.binaryexpression(lambda o: o.left(a).operator("+").right(n))))
print(f"{SystemVerilog.Text.ToText(total):30}", simplify(total, values).evaluate())
print("inferred:", Assertions.infer(delays[0], {"a": Domains.Anything, "b": Domains.Anything}).name())

a ##1 b                        the root is not a literal: SystemVerilog Assertions delaysequence is not simplified
a and b                        the root is not a literal: SystemVerilog Assertions binarysequence is not simplified
a |=> b                        the root is not a literal: SystemVerilog Assertions implicationproperty is not simplified
@(posedge clk) disable iff (rst) req |=> ack ##[1:3] done the root is not a literal: SystemVerilog Assertions propertyspec is not simplified
a + n                          32'sb00000000000000000000000000000011
inferred: any


<a id="T08.15.7"></a>
## 08.15.7 · Its editions are data

Assertions arrived with SystemVerilog, so the dialect covers IEEE 1800's five editions and none of Verilog's, each with
the kinds it has
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):
IEEE 1800-2009 added the strength (`strong(...)`, `weak(...)`), abort (`accept_on`, `reject_on`, ...) and case
properties:

In [8]:
singleton = store.Instances.Expressions.SystemVerilog.Assertions
kinds = {branch.schema: branch.name for branch in store.registered(Assertions.union).branches}
for designation, edition in Standards.standards(singleton).items():
    print(f"{designation:19}", f"{Standards.tags(edition)[1]:19}", len(Standards.kinds(edition)), "kinds")
first, second = (Standards.kinds(Standards.find(singleton, label)) for label in ("SystemVerilog-2005", "SystemVerilog-2009"))
print("added in IEEE 1800-2009:", ", ".join(kinds[k] for k in second if k not in first))

IEEE Std 1800-2005  SystemVerilog-2005  14 kinds
IEEE Std 1800-2009  SystemVerilog-2009  17 kinds
IEEE Std 1800-2012  SystemVerilog-2012  17 kinds
IEEE Std 1800-2017  SystemVerilog-2017  17 kinds
IEEE Std 1800-2023  SystemVerilog-2023  17 kinds
added in IEEE 1800-2009: strengthproperty, abortproperty, caseproperty


## What you learned

- **Sequences and properties are terms of `SystemVerilog Assertions`**: delays and cycle ranges, repetitions, the
  sequence and property operators, a property's clock and reset; their simplest operands are expressions.
- **The printer places parentheses by the standard's precedence**, from the tree, which holds none.
- **A call's arguments hold them**, so a named sequence or property is given one; its host validates the tree, an
  operator outside its kind's choices refused.
- **No clock cycle is simulated**: each kind is left in the residual, named, while an expression among them still folds; its
  editions are IEEE 1800's, three kinds since IEEE 1800-2009.

Next, what a randomized object's values may be: [SystemVerilog Constraints](08_16_SystemVerilog_Constraints.ipynb).

<!-- nav -->
[← 8.14 · SystemVerilog Patterns](08_14_SystemVerilog_Patterns.ipynb) · [Home](../../README.md) · [8.16 · SystemVerilog Constraints →](08_16_SystemVerilog_Constraints.ipynb)